# Part 3 : Cortex Code (CoCo) 活用例

> **CoCo の開き方:** Snowsight の右下の **CoCo アイコン（吹き出し）** をクリックするとチャットパネルが開きます。

In [ ]:
%%sql -r dataframe_3_1
-- コンテキスト設定
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE AI_HANDSON_ADAPTIVE_WH;
USE DATABASE AI_HANDSON_DB;
USE SCHEMA ANALYTICS;

---
## 1. コードの説明・レビュー

CoCoに既存のコードを読み解いてもらいます。
他人が書いたSQLを引き継いだときなど、複雑なクエリの意味を理解したいときに便利です。

1. 下の SQL セルを **クリックして選択** する(もしくはセル右上の星マークをクリックする)
2. CoCo チャットで以下のように聞いてみましょう

```
このSQLを説明して
```

> **ポイント:** CoCo はアクティブなセルのコードを読み取り、処理の流れ・各CTEの役割・最終出力を日本語で解説してくれます。

In [ ]:
%%sql -r dataframe_3_2
-- この SQL が何をしているか、CoCo に聞いてみましょう
WITH post_engagement AS (
    SELECT
        p.product_id,
        p.post_id,
        p.likes,
        p.comments,
        p.likes + p.comments * 2 AS engagement_score,
        f.has_person,
        f.location,
        f.product_usage
    FROM posts p
    JOIN image_features f
      ON p.image_path = f.image_file
),
product_summary AS (
    SELECT
        pe.product_id,
        pr.product_name,
        pr.category,
        COUNT(*)                          AS post_count,
        ROUND(AVG(pe.engagement_score),1) AS avg_engagement,
        SUM(CASE WHEN pe.has_person THEN 1 ELSE 0 END) AS with_person,
        SUM(CASE WHEN NOT pe.has_person THEN 1 ELSE 0 END) AS without_person
    FROM post_engagement pe
    JOIN products pr ON pe.product_id = pr.product_id
    GROUP BY pe.product_id, pr.product_name, pr.category
),
sales_summary AS (
    SELECT
        product_id,
        SUM(units_sold)   AS total_units,
        SUM(sales_amount) AS total_sales
    FROM daily_sales
    GROUP BY product_id
)
SELECT
    ps.product_name,
    ps.category,
    ps.post_count,
    ps.avg_engagement,
    ps.with_person,
    ps.without_person,
    ss.total_units,
    ss.total_sales,
    ROUND(ss.total_sales / NULLIF(ps.post_count, 0), 0) AS sales_per_post
FROM product_summary ps
JOIN sales_summary ss ON ps.product_id = ss.product_id
ORDER BY sales_per_post DESC;

---
## 2. 自然言語で SQL 生成

CoCo にやりたいことを自然言語で伝えるだけで、**テーブル構造を自分で探索して SQL を生成** してくれます。
テーブル名やカラム名を覚えていなくても大丈夫です。

### やってみよう

次のブランクセルを選択し、CoCo チャットで以下のように聞いてみましょう。

```
AI_HANDSON_DB.ANALYTICS にあるテーブルを使って、
商品カテゴリ別・週別のエンゲージメント率（いいね数÷投稿数）を集計するSQLを書いて
```

> **観察ポイント:**
> - CoCo がまず `SHOW TABLES` や `DESCRIBE TABLE` でテーブル構造を調べに行く
> - カラム名・型を確認してから SQL を組み立てる
> - 結果の SQL をそのままセルに貼って実行できる

### もう1つ試してみよう

```
人物が写っている投稿と写っていない投稿で、
平均いいね数と平均売上金額にどれくらい差があるか比較するSQLを書いて
```

> **ポイント:** テーブル名を指定していなくても、CoCo は `posts`・`image_features`・`daily_sales` を
> 自動的に見つけて JOIN してくれます。

In [ ]:
%%sql -r dataframe_3_3


---
## 3. エラー修正

以下は「商品カテゴリ × 投稿プラットフォーム別に、エンゲージメントと売上を集計する」SQLですが、**複数の間違いが含まれています**。

まず実行してエラーを確認し、CoCo に修正してもらいましょう。

In [ ]:
%%sql -r dataframe_3_4
-- 商品カテゴリ別・投稿プラットフォーム別のエンゲージメントと売上を分析
-- platform: 投稿先SNS（Instagram / X など）
-- total_revenue: 単価 × 販売数量 の合計
-- person_posts: 人物が写っている投稿の件数
SELECT
    pr.category,
    p.platform,
    COUNT(p.post_id)                            AS post_count,
    ROUND(AVG(p.likes), 1)                      AS avg_likes,
    SUM(ds.units_sold)                          AS total_units,
    ROUND(SUM(ds.price * ds.units_sold), 0)     AS total_revenue,
    SUM(CASE WHEN f.has_person THEN 1 ELSE 0 END) AS person_posts
FROM posts p
JOIN products pr       ON p.product_id = pr.product_id
JOIN daily_sales ds    ON p.product_id = ds.product_id
JOIN image_features f  ON p.post_id = f.post_id
GROUP BY pr.category, p.platform
ORDER BY total_revenue DESC;

**エラーを修正** をクリックしてください。


> **解答: 仕込まれたエラー 3 箇所**
>
> | # | エラー箇所 | 原因 | 修正方法 |
> |---|---|---|---|
> | 1 | `p.platform` | `posts` に `platform` は存在しない。`sns_mentions` のカラム | `sns_mentions` を JOIN して参照するか、カラムを削除 |
> | 2 | `ds.price * ds.units_sold` | `daily_sales` に `price` は存在しない。`products` のカラム | `pr.price * ds.units_sold` にするか、`ds.sales_amount` を使う |
> | 3 | `p.post_id = f.post_id` | `image_features` に `post_id` は存在しない | `p.image_path = f.image_file` に変更 |

---
## 4. テストデータ生成

CoCo はテーブル定義と既存データの文脈を理解して、**整合性のあるサンプルデータ**を自動生成できます。
ここでは CoCo に `.sql` ファイルを作らせ、インフルエンサー関連のサンプルデータを用意して既存データと JOIN 分析を行います。


### シナリオ

既存の `POSTS` テーブルには投稿データがありますが、**誰が投稿したか**の情報がありません。
インフルエンサーのマスタテーブルと、投稿との紐づけテーブルを新規作成して、
「どんなインフルエンサーの投稿がエンゲージメントや売上に貢献しているか」を分析しましょう。

### やってみよう

CoCo チャットで以下のように依頼してみましょう。

```
インフルエンサー別の投稿効果を分析したいが、まだデータがないのでサンプルデータから作りたい。
AI_HANDSON_DB.ANALYTICS の POSTS テーブルと JOIN できる形で、以下を含む SQL ファイル「create_influencer_data.sql」を作成して。

- インフルエンサーのマスタテーブルの作成とサンプルデータの投入
- 投稿との紐づけテーブルの作成とサンプルデータの投入
- 作成したデータと既存の投稿・商品データを JOIN した確認用の集計クエリ
- 末尾にクリーンアップ用の DROP TABLE（コメントアウト）
```

> **観察ポイント:**
> - カラム定義を指定しなくても、CoCo が既存テーブルの構造を読み取って適切なカラムを設計する
> - 左のファイル一覧に `create_influencer_data.sql` が現れる
> - ファイルを開いて、CoCo がどんなカラムやデータを設計したか確認してみましょう
> - 末尾にコメントアウトされた DROP TABLE 文があるので、ハンズン後の後片付けにも使える

ファイルが作成されたら、左サイドバーから **`create_influencer_data.sql` を開いて実行（▶）** してください。

**補足**
- sqlは人によって出てくるものが変わってくるので、適宜確認しながら実行してください。
- この様に長いコードを作成する時は1回でうまくいかないことも少なくなくないです。その場合CoCoにエラー修正を指示しながら進めてください。

---
## 5. ドキュメント生成

CoCo にスキーマのメタデータを読ませて、**テーブル定義書を自動生成** させます。
テーブルが増えるたびに手作業でドキュメントを更新する手間がなくなります。

### やってみよう

CoCo チャットで以下のように依頼してみましょう。

```
AI_HANDSON_DB.ANALYTICS スキーマの全テーブルについて、テーブル定義書を Markdown ファイルで新規作成して。
各テーブルごとに以下を含めて:
- テーブルの説明
- カラム一覧（カラム名、型、説明）
- 他テーブルとのリレーション
```

---
## 6. Streamlit アプリ作成（Plan モード）

ここでは CoCo の **Plan モード** を使って、設計 → 承認 → 実装 の流れを体験します。

Plan モードは、CoCo に「まず計画を立てさせてから実装に入る」ワークフローです。
複雑なタスクでいきなりコードを書き始めるのではなく、
**何をどう作るかを先に確認してから進められる**のがポイントです。

### Plan モードの使い方

1. CoCo チャットの入力欄の左にある **電球アイコン** をクリック
2. **「Plan mode」** をオンにする（トグルが青になる）
3. プロンプトを入力して送信する

> Plan モードがオンの間、CoCo は **コードの変更を行わず、計画だけを提示** します。
> 計画を確認して「OK」「進めて」と返すと、実装フェーズに移ります。

### やってみよう

**Plan モードをオンにしてから**、以下を入力してください。

```
AI_HANDSON_DB.ANALYTICS のデータを使って、
インフルエンサー施策の効果を分析する Streamlit ダッシュボードを作って。

以下の要素を含めたい:
- 商品別の売上サマリー（テーブル）
- 投稿のエンゲージメント推移（折れ線グラフ）
- 人物あり/なし写真の効果比較（棒グラフ）
- サイドバーでカテゴリをフィルタできるようにする
```

### 進め方

1. CoCo が **計画（ファイル構成・各コンポーネントの設計）** を提示する
2. 内容を確認する。修正したい点があれば、たとえば:
   - 「グラフの種類を円グラフに変えて」
   - 「日付でのフィルタも追加して」
3. 問題なければ **「OK、進めて」** と返す
4. CoCo が Plan モードを解除して **実装を開始** する

> **ポイント:**
> - Plan モードでは CoCo はファイルの読み取りだけ行い、書き込みはしません
> - 計画に納得してから実装に入れるので、手戻りが少なくなります
> - 複数ファイルにまたがる変更でも、全体像を先に確認できます

### 作成されたアプリを確認する

CoCo が実装を完了すると、ワークスペースに Streamlit アプリのファイルが作成されます。

1. 左サイドバーのファイル一覧に `.py` ファイルが追加されていることを確認
2. ファイルを開いてコードの中身を確認
3. Streamlit アプリの **展開 ボタン（▶）** をクリックして起動

---
## 7. App Runtime（デモ紹介）